#  ETL Pipeline — Módulo Prático
## Parte 2: Escalando com Apache Spark + PySpark
### Para Grandes Volumes de Dados

---

> **Objetivo:** Reproduzir o mesmo pipeline ETL da Parte 1 usando **PySpark**, demonstrando como lidar com grandes volumes de dados de forma distribuída.

---

## 🆚 Por que PySpark em vez de Pandas?

| Característica | Pandas | PySpark |
|---|---|---|
| **Volume máximo** | ~GBs (limitado pela RAM) | Petabytes (distribuído) |
| **Processamento** | Single-machine | Cluster distribuído |
| **Execução** | Eager (imediata) | Lazy (otimizada) |
| **Tolerância a falhas** | Nenhuma | Lineage + recomputação |
| **Paralelismo** | Limitado (GIL Python) | Nativo e automático |
| **Ideal para** | Análise exploratória | Produção em escala |

---

## 📌 Estrutura desta Demo

| Etapa | Descrição |
|---|---|
| **1. Setup Spark** | Instalação e configuração da SparkSession |
| **2. Ingestão (Bronze)** | Leitura de dados via requests → Spark DataFrame |
| **3. Limpeza (Silver)** | Transformações com Spark SQL e DSL |
| **4. Enriquecimento (Gold)** | Agregações distribuídas |
| **5. Qualidade de Dados** | Validações com Spark nativo |
| **6. Comparativo** | Pandas vs PySpark — quando usar cada um |

---
## ⚙️ Etapa 1 — Setup do Apache Spark no Google Colab

In [ ]:
# ============================================================
# ETAPA 1.1 — INSTALAÇÃO DO PYSPARK
# ============================================================
# O Google Colab pode já ter o PySpark instalado (via dataproc-spark-connect).
# Verificamos a versão disponível e instalamos apenas se necessário,
# sempre respeitando o que o ambiente já oferece.

import subprocess, sys

def get_installed_pyspark_version():
    try:
        import pyspark
        return pyspark.__version__
    except ImportError:
        return None

installed = get_installed_pyspark_version()

if installed:
    print(f" PySpark {installed} já está instalado — nenhuma ação necessária.")
else:
    print(" PySpark não encontrado. Instalando...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyspark", "--quiet"])
    installed = get_installed_pyspark_version()
    print(f" PySpark {installed} instalado com sucesso!")

# Dependências auxiliares
!pip install requests colorama tabulate --quiet

print(f"\n Ambiente pronto — PySpark versão: {installed}")

In [ ]:
# ============================================================
# ETAPA 1.2 — CRIAÇÃO DA SPARKSESSION
# ============================================================
# SparkSession é o ponto de entrada para toda interação com o Spark.
# É análoga à 'conexão' com um banco de dados.
#
#  NOTA DE COMPATIBILIDADE:
# O Google Colab pode ter Spark 3.x ou 4.x dependendo do ambiente.
# Este código detecta a versão e aplica as configurações corretas.

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, DoubleType, TimestampType, LongType
)
from pyspark.sql.window import Window
import pyspark
import pandas as pd
import requests
import json
import os
import hashlib
import json as _json
from datetime import datetime, timezone
from colorama import Fore, Style, init
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')

init(autoreset=True)

SPARK_VERSION = tuple(int(x) for x in pyspark.__version__.split('.')[:2])
print(f"🔍 Versão detectada: PySpark {pyspark.__version__}")

# ── Configuração da SparkSession compatível com Spark 3.x e 4.x ──
builder = (
    SparkSession.builder
    .appName("ETL_Pipeline_Demo")
    .config("spark.sql.adaptive.enabled", True)
    .config("spark.sql.adaptive.coalescePartitions.enabled", True)
    .config("spark.driver.memory", "2g")
)

# eagerEval existe apenas no Spark 3.x (removido no 4.x)
if SPARK_VERSION < (4, 0):
    builder = (
        builder
        .config("spark.sql.repl.eagerEval.enabled", True)
        .config("spark.sql.repl.eagerEval.maxNumRows", 10)
    )

spark = builder.getOrCreate()

# Reduz logs verbosos
spark.sparkContext.setLogLevel("WARN")

print(f"{Fore.GREEN} SparkSession criada com sucesso!")
print(f"   Versão do Spark : {spark.version}")
print(f"   App Name        : {spark.sparkContext.appName}")
print(f"   Modo            : {spark.sparkContext.master}")

# ── Paths do Data Lake ──
BASE_PATH    = "/content/spark_data_lake"
BRONZE_PATH  = f"{BASE_PATH}/bronze"
SILVER_PATH  = f"{BASE_PATH}/silver"
GOLD_PATH    = f"{BASE_PATH}/gold"
QUALITY_PATH = f"{BASE_PATH}/quality_reports"

for p in [BRONZE_PATH, SILVER_PATH, GOLD_PATH, QUALITY_PATH]:
    os.makedirs(p, exist_ok=True)

INGESTION_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
print(f"\n{Fore.CYAN} Data Lake Spark: {BASE_PATH}")

In [ ]:
# ============================================================
# ETAPA 1.3 — CONCEITO: LAZY EVALUATION
# ============================================================
#  Conceito fundamental do Spark: LAZY EVALUATION
#
# O Spark NÃO executa as transformações imediatamente.
# Ele constrói um 'plano lógico' (DAG) e só executa quando
# chamamos uma 'action' (como .show(), .count(), .write())
#
# Isso permite ao Spark OTIMIZAR todo o pipeline antes de executar.
#
# TRANSFORMATIONS (lazy) → filter, select, join, groupBy, withColumn
# ACTIONS (eager)        → show, count, collect, write, toPandas

print(f"{Fore.YELLOW} DEMONSTRAÇÃO DE LAZY EVALUATION\n")

# Criando um DataFrame simples
demo_data = [(1, 'Alice', 1000), (2, 'Bob', 2000), (3, 'Carol', 1500)]
df_demo = spark.createDataFrame(demo_data, ['id', 'name', 'salary'])

# Transformações — ainda NÃO executadas
df_filtered = df_demo.filter(F.col('salary') > 1200)
df_doubled  = df_filtered.withColumn('salary_doubled', F.col('salary') * 2)

print("  → Definindo filtro (ainda não executado — LAZY)")
print("  → Definindo transformação (ainda não executado — LAZY)")
print(f"  → Plano de execução:")
df_doubled.explain(mode='simple')  # Mostra o plano sem executar

print(f"\n  → Chamando .show() — ACTION: Spark executa tudo agora!")
df_doubled.show()

---
## 🥉 Etapa 2 — Camada BRONZE: Ingestão com Spark

Em produção real, o Spark leria de **S3, HDFS, Kafka, Delta Lake, etc.**
Aqui, ingerimos via API REST e convertemos para Spark DataFrame.

In [ ]:
# ============================================================
# ETAPA 2.1 — FUNÇÃO DE INGESTÃO COM SPARK
# ============================================================

def ingest_to_spark(endpoint: str, entity_name: str) -> 'pyspark.sql.DataFrame':
    """
    Ingere dados de uma API e cria um Spark DataFrame.
    Em produção: substituir por spark.read.parquet(s3://...) etc.
    """
    print(f"{Fore.CYAN} Ingerindo [{entity_name}] para Spark...")

    # Coleta via API (em produção seria leitura direta de storage)
    response = requests.get(endpoint, timeout=10)
    response.raise_for_status()
    raw_data = response.json()

    # Pandas como intermediário para ingestão inicial
    df_pandas = pd.DataFrame(raw_data)

    # Converter para Spark DataFrame
    # O Spark infere o schema automaticamente a partir do Pandas
    df_spark = spark.createDataFrame(df_pandas)

    # Adicionar metadados de ingestão usando funções nativas do Spark
    df_spark = df_spark \
        .withColumn('_ingestion_timestamp', F.lit(INGESTION_TIMESTAMP)) \
        .withColumn('_source_url',          F.lit(endpoint)) \
        .withColumn('_source_system',       F.lit('jsonplaceholder_api')) \
        .withColumn('_load_date',           F.current_date()) \
        .withColumn('_record_id',           F.monotonically_increasing_id())

    # Salvar na Bronze como Parquet particionado
    output_path = f"{BRONZE_PATH}/{entity_name}"
    df_spark.write.mode('overwrite').parquet(output_path)

    record_count = df_spark.count()
    print(f"{Fore.GREEN}    {record_count} registros | {len(df_spark.columns)} colunas")
    print(f"    Salvo: {output_path} (Parquet)")

    return df_spark

print(f"{Fore.GREEN} Função de ingestão Spark definida!")

In [ ]:
# ============================================================
# ETAPA 2.1 — FUNÇÃO DE INGESTÃO COM SPARK
# ============================================================


def ingest_to_spark(endpoint: str, entity_name: str):
    print(f"{Fore.CYAN} Ingerindo [{entity_name}] para Spark...")

    response = requests.get(endpoint, timeout=10)
    response.raise_for_status()
    raw_data = response.json()

    # ── Achatar colunas aninhadas (dicts/listas) para JSON string ──
    # O Spark não consegue inferir schema de dicts Python heterogêneos.
    # Serializamos como string JSON e depois usamos from_json() para
    # recriar a estrutura de forma controlada dentro do Spark.
    flat_rows = []
    for record in raw_data:
        flat = {}
        for k, v in record.items():
            if isinstance(v, (dict, list)):
                flat[k] = _json.dumps(v)   # dict → string JSON
            else:
                flat[k] = v
        flat_rows.append(flat)

    df_pandas = pd.DataFrame(flat_rows)
    df_spark  = spark.createDataFrame(df_pandas)

    # ── Metadados de ingestão ──
    df_spark = (
        df_spark
        .withColumn('_ingestion_timestamp', F.lit(INGESTION_TIMESTAMP))
        .withColumn('_source_url',          F.lit(endpoint))
        .withColumn('_source_system',       F.lit('jsonplaceholder_api'))
        .withColumn('_load_date',           F.current_date())
        .withColumn('_record_id',           F.monotonically_increasing_id())
    )

    output_path = f"{BRONZE_PATH}/{entity_name}"
    df_spark.write.mode('overwrite').parquet(output_path)

    record_count = df_spark.count()
    print(f"{Fore.GREEN}    {record_count} registros | {len(df_spark.columns)} colunas")
    print(f"    Salvo: {output_path} (Parquet)")

    return df_spark

# ── Re-executar a ingestão ──
print(f"{Fore.YELLOW}{'='*60}")
print(f"{Fore.YELLOW}  PIPELINE DE INGESTÃO — SPARK")
print(f"{Fore.YELLOW}{'='*60}\n")

BASE_URL = "https://jsonplaceholder.typicode.com"

entities = {
    'users':    f"{BASE_URL}/users",
    'posts':    f"{BASE_URL}/posts",
    'todos':    f"{BASE_URL}/todos",
    'comments': f"{BASE_URL}/comments",
}

bronze_spark = {}
for entity, url in entities.items():
    bronze_spark[entity] = ingest_to_spark(url, entity)
    print()

print(f"{Fore.GREEN} Ingestão Spark concluída!")

In [ ]:
# ============================================================
# ETAPA 2.3 — INSPEÇÃO DO SCHEMA E PLANO DE EXECUÇÃO
# ============================================================
# No Spark, sempre inspecionamos o schema antes de transformar.

print(f"{Fore.YELLOW} SCHEMA DOS DADOS BRUTOS (Bronze)\n")

for entity, df in bronze_spark.items():
    print(f"{Fore.CYAN}{'─'*50}")
    print(f"{Fore.CYAN}  {entity.upper()}")
    df.printSchema()
    df.show(3, truncate=50)

---
## 🥈 Etapa 3 — Camada SILVER com PySpark

PySpark oferece duas formas de escrever transformações:
- **DSL (Domain Specific Language)**: `df.filter(col('age') > 18)` — pythônico
- **Spark SQL**: `spark.sql("SELECT * FROM df WHERE age > 18")` — familiar para quem conhece SQL

> 💡 Ambas geram o **mesmo plano de execução** — use a que for mais legível para sua equipe.

In [ ]:
# ============================================================
# ETAPA 3.1 — SILVER: USERS (com DSL e SQL)
# ============================================================
# Como address/company foram serializados como JSON string na Bronze,
# usamos get_json_object() para extrair os campos aninhados.

print(f"{Fore.CYAN} Processando USERS — Spark Silver\n")

df_users_raw = bronze_spark['users']

print("  → Expandindo campos aninhados via get_json_object (address, company)...")
df_users_raw.printSchema()

# ── 1. Flatten com get_json_object (para colunas JSON string) ──
df_users_silver = df_users_raw.select(
    F.col('id').cast(IntegerType()).alias('user_id'),
    F.col('name'),
    F.col('username'),
    F.col('email'),
    F.col('phone'),
    F.col('website'),
    # Endereço — extraído do JSON string
    F.get_json_object(F.col('address'), '$.street').alias('addr_street'),
    F.get_json_object(F.col('address'), '$.suite').alias('addr_suite'),
    F.get_json_object(F.col('address'), '$.city').alias('addr_city'),
    F.get_json_object(F.col('address'), '$.zipcode').alias('addr_zipcode'),
    F.get_json_object(F.col('address'), '$.geo.lat').cast(DoubleType()).alias('geo_lat'),
    F.get_json_object(F.col('address'), '$.geo.lng').cast(DoubleType()).alias('geo_lng'),
    # Empresa — extraído do JSON string
    F.get_json_object(F.col('company'), '$.name').alias('company_name'),
    F.get_json_object(F.col('company'), '$.catchPhrase').alias('company_catchphrase'),
    F.get_json_object(F.col('company'), '$.bs').alias('company_bs'),
    # Metadados
    F.col('_ingestion_timestamp'),
    F.col('_source_system'),
)

# ── 2. Normalização ──
print("  → Normalizando campos de texto...")
df_users_silver = df_users_silver \
    .withColumn('email',    F.trim(F.lower(F.col('email')))) \
    .withColumn('username', F.trim(F.lower(F.col('username')))) \
    .withColumn('name',     F.initcap(F.trim(F.col('name')))) \
    .withColumn('phone',    F.regexp_replace(F.col('phone'), r'[^\d\+\-\(\) x]', ''))

# ── 3. Deduplicação ──
print("  → Removendo duplicatas por user_id...")
window_spec = Window.partitionBy('user_id').orderBy(F.col('_ingestion_timestamp').desc())
df_users_silver = df_users_silver \
    .withColumn('row_num', F.row_number().over(window_spec)) \
    .filter(F.col('row_num') == 1) \
    .drop('row_num')

# ── 4. Metadados Silver ──
df_users_silver = df_users_silver \
    .withColumn('_silver_processed_at', F.current_timestamp()) \
    .withColumn('_pipeline_version',    F.lit('v1.0_spark'))

# ── 5. Cache + salvar ──
df_users_silver.cache()
df_users_silver.write.mode('overwrite').parquet(f"{SILVER_PATH}/users")

print(f"\n{Fore.GREEN}   Users Silver: {df_users_silver.count()} registros")
df_users_silver.show(5, truncate=40)

In [ ]:
# ============================================================
# ETAPA 3.2 — USANDO SPARK SQL
# ============================================================
# Demonstrando o mesmo resultado com SQL puro no Spark
#
# 💡 NOTA: QUALIFY é suportado apenas no Spark 3.3+.
# Usamos uma subquery com ROW_NUMBER para garantir compatibilidade.

print(f"{Fore.CYAN} Processando POSTS com Spark SQL\n")

df_posts_raw = bronze_spark['posts']

# Registrar como Temp View para usar SQL
df_posts_raw.createOrReplaceTempView("posts_raw")

# Usar Spark SQL — exatamente como um banco de dados!
# Deduplicação via subquery + ROW_NUMBER (compatível com Spark 3.x e 4.x)
df_posts_silver = spark.sql("""
    SELECT post_id, user_id, title, body,
           title_word_count, body_word_count, body_length,
           _ingestion_timestamp, _source_system, _silver_processed_at
    FROM (
        SELECT
            CAST(id AS INT)                          AS post_id,
            CAST(userId AS INT)                      AS user_id,
            INITCAP(TRIM(title))                     AS title,
            TRIM(REGEXP_REPLACE(body, '\\n', ' '))   AS body,
            SIZE(SPLIT(TRIM(title), ' '))            AS title_word_count,
            SIZE(SPLIT(TRIM(body),  ' '))            AS body_word_count,
            LENGTH(body)                             AS body_length,
            _ingestion_timestamp,
            _source_system,
            current_timestamp()                      AS _silver_processed_at,
            ROW_NUMBER() OVER (
                PARTITION BY id
                ORDER BY _ingestion_timestamp DESC
            ) AS rn
        FROM posts_raw
    ) t
    WHERE rn = 1
""")

# Persistir e salvar
df_posts_silver.cache()
df_posts_silver.write.mode('overwrite').parquet(f"{SILVER_PATH}/posts")

print(f"{Fore.GREEN}   Posts Silver (via Spark SQL): {df_posts_silver.count()} registros")
df_posts_silver.show(5, truncate=50)

In [ ]:
# ============================================================
# ETAPA 3.3 — SILVER: TODOS
# ============================================================

print(f"{Fore.CYAN} Processando TODOS — Spark Silver\n")

df_todos_raw = bronze_spark['todos']
df_todos_raw.createOrReplaceTempView("todos_raw")

df_todos_silver = spark.sql("""
    SELECT
        CAST(id AS INT)       AS todo_id,
        CAST(userId AS INT)   AS user_id,
        INITCAP(TRIM(title))  AS title,
        CAST(completed AS BOOLEAN) AS completed,
        CASE WHEN completed = true THEN 'COMPLETED' ELSE 'PENDING' END AS status,
        _ingestion_timestamp,
        _source_system,
        current_timestamp() AS _silver_processed_at
    FROM todos_raw
""")

df_todos_silver.cache()
df_todos_silver.write.mode('overwrite').parquet(f"{SILVER_PATH}/todos")

print(f"{Fore.GREEN}   Todos Silver: {df_todos_silver.count()} registros")
df_todos_silver.show(5)

print(f"\n{Fore.YELLOW} Distribuição de status:")
df_todos_silver.groupBy('status').count().show()

---
## 🥇 Etapa 4 — Camada GOLD com PySpark

Joins distribuídos e agregações em larga escala.

> 💡 **Otimização Spark:** Para joins com tabelas pequenas, usar **Broadcast Join** evita shuffle de dados pela rede, melhorando drasticamente a performance.

In [ ]:
# ============================================================
# ETAPA 4.1 — GOLD: PERFIL COMPLETO DO USUÁRIO
# ============================================================

print(f"{Fore.YELLOW} Construindo camada GOLD com Spark...\n")

# ── Agregações distribuídas ──
posts_agg_spark = df_posts_silver.groupBy('user_id').agg(
    F.count('post_id').alias('total_posts'),
    F.round(F.avg('title_word_count'), 2).alias('avg_title_words'),
    F.round(F.avg('body_length'), 2).alias('avg_body_length'),
    F.sum('body_length').alias('total_body_length'),
)

todos_agg_spark = df_todos_silver.groupBy('user_id').agg(
    F.count('todo_id').alias('total_tasks'),
    F.sum(F.col('completed').cast('int')).alias('completed_tasks'),
    F.sum((~F.col('completed')).cast('int')).alias('pending_tasks'),
)
todos_agg_spark = todos_agg_spark.withColumn(
    'completion_rate_pct',
    F.round(F.col('completed_tasks') / F.col('total_tasks') * 100, 1)
)

# ── Join com Broadcast Hint (otimização para tabelas pequenas) ──
# Quando uma tabela é pequena (<= 10MB), Broadcast Join evita shuffle
print("  → Executando joins distribuídos (com Broadcast Hint)...")

users_select = df_users_silver.select(
    'user_id', 'name', 'email', 'username',
    'addr_city', 'addr_zipcode', 'company_name', 'geo_lat', 'geo_lng'
)

df_gold_profile = (
    users_select
    .join(F.broadcast(posts_agg_spark), on='user_id', how='left')   # Broadcast: posts_agg é pequeno
    .join(F.broadcast(todos_agg_spark), on='user_id', how='left')   # Broadcast: todos_agg é pequeno
)

# ── Classificação com CASE WHEN (equivalente ao Spark SQL) ──
df_gold_profile = df_gold_profile.withColumn(
    'engagement_level',
    F.when(
        (F.col('completion_rate_pct') >= 70) & (F.col('total_posts') >= 10), 'HIGH'
    ).when(
        (F.col('completion_rate_pct') >= 40) | (F.col('total_posts') >= 5), 'MEDIUM'
    ).otherwise('LOW')
)

# ── Metadados Gold ──
df_gold_profile = df_gold_profile \
    .withColumn('_gold_processed_at', F.current_timestamp())

# ── Persistir e salvar ──
df_gold_profile.cache()
df_gold_profile.write.mode('overwrite').parquet(f"{GOLD_PATH}/user_profile")

print(f"{Fore.GREEN}\n   Gold User Profile: {df_gold_profile.count()} registros")
df_gold_profile.show(10, truncate=40)

In [ ]:
# ============================================================
# ETAPA 4.2 — VER O PLANO DE EXECUÇÃO FÍSICO
# ============================================================
# O plano físico mostra como o Spark vai REALMENTE executar a query.
# 'BroadcastHashJoin' confirma que o Broadcast Hint funcionou!

print(f"{Fore.YELLOW} PLANO DE EXECUÇÃO FÍSICO (Broadcast Join):")
df_gold_profile.explain(mode='formatted')

In [ ]:
# ============================================================
# ETAPA 4.3 — GOLD: ANÁLISE POR CIDADE
# ============================================================

df_gold_profile.createOrReplaceTempView("gold_profile")

df_gold_city = spark.sql("""
    SELECT
        addr_city,
        COUNT(*) AS total_users,
        SUM(total_posts) AS total_posts,
        ROUND(AVG(completion_rate_pct), 2) AS avg_completion_pct,
        SUM(CASE WHEN engagement_level = 'HIGH' THEN 1 ELSE 0 END) AS high_engagement_users,
        SUM(CASE WHEN engagement_level = 'LOW'  THEN 1 ELSE 0 END) AS low_engagement_users
    FROM gold_profile
    GROUP BY addr_city
    ORDER BY avg_completion_pct DESC
""")

df_gold_city.write.mode('overwrite').parquet(f"{GOLD_PATH}/city_analysis")

print(f"{Fore.GREEN} Gold - Análise por Cidade:")
df_gold_city.show()

# KPIs globais
kpis = df_gold_profile.agg(
    F.countDistinct('user_id').alias('unique_users'),
    F.sum('total_posts').alias('total_posts'),
    F.round(F.avg('completion_rate_pct'), 1).alias('avg_completion_pct'),
    F.sum((F.col('engagement_level') == 'HIGH').cast('int')).alias('high_engagement_count')
).collect()[0]

print(f"\n{Fore.YELLOW} KPIs GLOBAIS (Spark):")
print(f"    Usuários únicos    : {kpis['unique_users']}")
print(f"    Total de posts     : {kpis['total_posts']}")
print(f"    Taxa conclusão avg : {kpis['avg_completion_pct']}%")
print(f"    Alta engajamento   : {kpis['high_engagement_count']} usuários")

---
## ✅ Etapa 5 — Qualidade de Dados com PySpark

As mesmas verificações da Parte 1, agora de forma distribuída.
Cada verificação gera um **Spark job** que roda em paralelo.

In [ ]:
# ============================================================
# ETAPA 5.1 — FRAMEWORK DE QUALIDADE PARA SPARK
# ============================================================

class SparkDataQualityCheck:
    """Framework de qualidade de dados para Spark DataFrames."""

    def __init__(self, df, entity_name: str, layer: str):
        self.df = df
        self.entity_name = entity_name
        self.layer = layer
        self.results = []
        self.passed = self.failed = self.warnings = 0
        # Cache para evitar recomputação
        self.df.cache()
        self._total_rows = df.count()

    def _add_result(self, check_name, dimension, passed, details, severity='ERROR'):
        if passed:
            self.passed += 1
        elif severity == 'WARNING':
            self.warnings += 1
        else:
            self.failed += 1
        self.results.append({
            'check': check_name, 'dimension': dimension,
            'status': ' PASS' if passed else ('  WARN' if severity == 'WARNING' else ' FAIL'),
            'details': details, 'severity': severity
        })

    def check_not_null(self, columns, severity='ERROR'):
        for col in columns:
            if col not in self.df.columns:
                self._add_result(f'not_null:{col}', 'COMPLETENESS', False,
                                 f'Coluna {col} não existe', severity)
                continue
            null_count = self.df.filter(F.col(col).isNull()).count()
            null_pct   = null_count / self._total_rows * 100
            self._add_result(f'not_null:{col}', 'COMPLETENESS', null_count == 0,
                             f'{null_count} nulos ({null_pct:.1f}%)', severity)
        return self

    def check_unique(self, columns, severity='ERROR'):
        for col in columns:
            if col not in self.df.columns:
                continue
            distinct_count = self.df.select(col).distinct().count()
            dup_count = self._total_rows - distinct_count
            self._add_result(f'unique:{col}', 'UNIQUENESS', dup_count == 0,
                             f'{dup_count} duplicatas', severity)
        return self

    def check_regex(self, column, pattern, description, severity='WARNING'):
        if column not in self.df.columns:
            return self
        invalid_count = self.df.filter(
            F.col(column).isNotNull() &
            ~F.col(column).rlike(pattern)
        ).count()
        self._add_result(f'regex:{column}', 'VALIDITY', invalid_count == 0,
                         f'{invalid_count} com formato inválido ({description})', severity)
        return self

    def check_range(self, column, min_val=None, max_val=None, severity='WARNING'):
        if column not in self.df.columns:
            return self
        cond = F.col(column).isNotNull()
        if min_val is not None:
            cond = cond & (F.col(column) < min_val)
        if max_val is not None:
            cond = cond | (F.col(column) > max_val)
        out_count = self.df.filter(cond).count()
        self._add_result(f'range:{column}', 'VALIDITY', out_count == 0,
                         f'{out_count} fora do intervalo [{min_val}, {max_val}]', severity)
        return self

    def check_referential_integrity(self, column, ref_df, ref_column, severity='ERROR'):
        if column not in self.df.columns:
            return self
        orphans = self.df.join(
            ref_df.select(ref_column).distinct(),
            self.df[column] == ref_df[ref_column], 'left_anti'
        ).count()
        self._add_result(f'referential:{column}→{ref_column}', 'CONSISTENCY',
                         orphans == 0, f'{orphans} registros órfãos', severity)
        return self

    def check_row_count(self, min_rows, severity='ERROR'):
        passed = self._total_rows >= min_rows
        self._add_result('row_count', 'COMPLETENESS', passed,
                         f'{self._total_rows} linhas (mín: {min_rows})', severity)
        return self

    def generate_report(self):
        total = self.passed + self.failed + self.warnings
        score = round(self.passed / total * 100, 1) if total > 0 else 0

        print(f"\n{'═'*60}")
        print(f"  QUALIDADE SPARK — {self.entity_name.upper()} [{self.layer}]")
        print(f"{'═'*60}")
        print(tabulate(
            [(r['status'], r['dimension'], r['check'], r['details']) for r in self.results],
            headers=['Status', 'Dimensão', 'Verificação', 'Detalhes'],
            tablefmt='rounded_outline'
        ))

        color = Fore.GREEN if score >= 80 else (Fore.YELLOW if score >= 60 else Fore.RED)
        print(f"\n{color} Score: {score}% |  {self.passed} |  {self.failed} |  {self.warnings}")

        report = {
            'entity': self.entity_name, 'layer': self.layer, 'engine': 'spark',
            'score': score, 'timestamp': datetime.now(timezone.utc).isoformat(),
            'summary': {'passed': self.passed, 'failed': self.failed, 'warnings': self.warnings},
            'checks': self.results
        }
        with open(f"{QUALITY_PATH}/{self.entity_name}_{self.layer}_spark_quality.json", 'w') as f:
            json.dump(report, f, indent=2, default=str)
        return report

print(f"{Fore.GREEN} Framework de qualidade Spark definido!")

In [ ]:
# ============================================================
# ETAPA 5.2 — EXECUÇÃO DAS VERIFICAÇÕES
# ============================================================

print(f"{Fore.YELLOW} Executando verificações de qualidade com Spark...\n")

quality_spark = {}

# Users Silver
quality_spark['users'] = (
    SparkDataQualityCheck(df_users_silver, 'users', 'SILVER')
    .check_row_count(min_rows=5)
    .check_not_null(['user_id', 'name', 'email', 'username'])
    .check_unique(['user_id', 'email'])
    .check_regex('email', r'^[\w\.\-]+@[\w\.\-]+\.\w+$', 'formato email', severity='WARNING')
    .check_range('geo_lat', min_val=-90, max_val=90, severity='WARNING')
    .generate_report()
)

# Posts Silver
quality_spark['posts'] = (
    SparkDataQualityCheck(df_posts_silver, 'posts', 'SILVER')
    .check_row_count(min_rows=50)
    .check_not_null(['post_id', 'user_id', 'title'])
    .check_unique(['post_id'])
    .check_referential_integrity('user_id', df_users_silver, 'user_id')
    .check_range('title_word_count', min_val=1, max_val=50)
    .generate_report()
)

# Gold Profile
quality_spark['gold_profile'] = (
    SparkDataQualityCheck(df_gold_profile, 'user_profile', 'GOLD')
    .check_not_null(['user_id', 'name', 'email', 'total_posts', 'total_tasks'])
    .check_unique(['user_id'])
    .check_range('completion_rate_pct', min_val=0, max_val=100)
    .generate_report()
)

---
## 📊 Etapa 6 — SIMULANDO GRANDE VOLUME DE DADOS

In [ ]:
# ============================================================
# BÔNUS — SIMULANDO GRANDE VOLUME DE DADOS
# ============================================================
# Demonstramos que o Spark suporta grandes volumes
# gerando um dataset sintético de 1 MILHÃO de linhas.

print(f"{Fore.YELLOW} BÔNUS: Processando 1 MILHÃO de registros com Spark...\n")

import time

# ── Gerar 1M de registros sintéticos com Spark ──
N = 1_000_000

df_large = spark.range(N).select(
    F.col('id'),
    (F.col('id') % 10_000).alias('user_id'),
    F.when(F.rand() > 0.5, True).otherwise(False).alias('is_active'),
    F.round(F.rand() * 10000, 2).alias('transaction_value'),
    F.date_add(F.lit('2024-01-01'), (F.rand() * 365).cast('int')).alias('transaction_date'),
    F.array(
        F.lit('category_a'), F.lit('category_b'), F.lit('category_c')
    ).getItem((F.rand() * 3).cast('int')).alias('category'),
)

# Medir tempo de processamento
start = time.time()

# Agregação distribuída em 1M de linhas
result = df_large.groupBy('category').agg(
    F.count('id').alias('total_transactions'),
    F.sum('transaction_value').alias('total_value'),
    F.round(F.avg('transaction_value'), 2).alias('avg_value'),
    F.sum(F.col('is_active').cast('int')).alias('active_count'),
).orderBy('category')

result.show()
elapsed = time.time() - start

print(f"{Fore.GREEN}⚡ {N:,} registros processados em {elapsed:.2f}s")
print(f"{Fore.CYAN}   Throughput: ~{int(N/elapsed):,} registros/segundo")
print(f"\n{Fore.YELLOW} Escale horizontalmente adicionando workers ao cluster!")

# ── Encerrar SparkSession ──
spark.stop()
print(f"\n{Fore.GREEN} SparkSession encerrada. Demo completa!")